# MTGFlow su PVGIS

Workflow guidato per preparare i CSV climatici, eseguire MTGFlow e analizzare gli score. Gli script restano la fonte riproducibile; il notebook li orchestra senza duplicare la logica.

Protocollo: **training 2005–2018**, **target/test 2019**, z-score e soglie calcolati esclusivamente sul training.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Avviare il notebook dalla root del repository o da notebooks/.')

# Modificare soltanto questo percorso.
PVGIS_DIR = Path(r'C:\path\to\pvgis_data')
OUTPUT_ROOT = ROOT / 'outputs' / 'pvgis_mtgflow'
PREPARED_DIR = OUTPUT_ROOT / 'prepared'
MANIFEST = PREPARED_DIR / 'manifest_shard_0000.csv'
DEVICE = 'cuda'

RUN_PREPARATION = False
RUN_SMOKE = False
RUN_FULL = False
print('Repository:', ROOT)
print('PVGIS:', PVGIS_DIR)

In [ ]:
expected_files = [PVGIS_DIR / f'piedmont_pvgis_{year}.nc' for year in range(2005, 2020)]
missing_files = [path for path in expected_files if not path.is_file()]
print(f'File trovati: {len(expected_files) - len(missing_files)}/{len(expected_files)}')
if missing_files:
    print('Primi file mancanti:', *missing_files[:5], sep='\n- ')
else:
    print('Dataset completo per il protocollo 2005–2018 -> 2019.')

## 1. Preparazione dei CSV

Non viene applicata normalizzazione stagionale: MTGFlow esegue internamente lo z-score training-only previsto dal protocollo.

In [ ]:
prepare_command = [
    sys.executable,
    str(ROOT / 'scripts' / 'prepare_pvgis_mtgflow.py'),
    '--pvgis-dir', str(PVGIS_DIR),
    '--out-dir', str(PREPARED_DIR),
    '--train-start', '2005',
    '--train-end', '2018',
    '--test-year', '2019',
]
print(' '.join(prepare_command))
if RUN_PREPARATION:
    if missing_files:
        raise FileNotFoundError('Mancano uno o più NetCDF richiesti.')
    subprocess.run(prepare_command, cwd=ROOT, check=True)
else:
    print('Preparazione non eseguita: impostare RUN_PREPARATION=True.')

In [ ]:
if MANIFEST.is_file():
    manifest = pd.read_csv(MANIFEST)
    print(f'Località preparate: {len(manifest):,}')
    display(manifest.head())
else:
    manifest = None
    print('Manifest non ancora disponibile:', MANIFEST)

## 2. Smoke test

Esegue una configurazione ridotta su una sola località. È un controllo tecnico e viene marcato nei metadati come configurazione custom.

In [ ]:
SMOKE_OUTPUT = OUTPUT_ROOT / 'smoke'
smoke_command = [
    sys.executable,
    str(ROOT / 'scripts' / 'run_pvgis_mtgflow.py'),
    '--manifest', str(MANIFEST),
    '--out-dir', str(SMOKE_OUTPUT),
    '--device', DEVICE,
    '--seed', '15',
    '--epochs', '1',
    '--max-locations', '1',
]
print(' '.join(smoke_command))
if RUN_SMOKE:
    if not MANIFEST.is_file():
        raise FileNotFoundError('Preparare prima il manifest.')
    subprocess.run(smoke_command, cwd=ROOT, check=True)
else:
    print('Smoke test non eseguito: impostare RUN_SMOKE=True.')

## 3. Run completa di riferimento

Usa finestra 60, stride 10, 40 epoche, due flow block, batch 256 e i seed 15–19. La directory di output deve essere nuova o vuota.

In [ ]:
REFERENCE_OUTPUT = OUTPUT_ROOT / 'reference_run'
reference_command = [
    sys.executable,
    str(ROOT / 'scripts' / 'run_pvgis_mtgflow.py'),
    '--manifest', str(MANIFEST),
    '--out-dir', str(REFERENCE_OUTPUT),
    '--device', DEVICE,
]
print(' '.join(reference_command))
if RUN_FULL:
    if not MANIFEST.is_file():
        raise FileNotFoundError('Preparare prima il manifest.')
    subprocess.run(reference_command, cwd=ROOT, check=True)
else:
    print('Run completa non eseguita: impostare RUN_FULL=True.')

## 4. Lettura e controllo dei CSV

Gli score dei seed restano separati. Il timestamp rappresenta `window_end`; il flag riguarda tutta la finestra `window_start`–`window_end`.

In [ ]:
ANALYSIS_ROOT = REFERENCE_OUTPUT
SEED = 15
scores_path = ANALYSIS_ROOT / f'seed_{SEED}' / 'anomaly_scores.csv'
metadata_path = ANALYSIS_ROOT / 'run_metadata.json'

if scores_path.is_file():
    scores = pd.read_csv(scores_path, parse_dates=['window_start', 'window_end', 'timestamp'])
    print(f'Score caricati: {len(scores):,}')
    display(scores.head())
    summary = scores.groupby('location').agg(
        n_windows=('is_anomaly', 'size'),
        n_anomaly=('is_anomaly', 'sum'),
        mean_score=('anomaly_score', 'mean'),
    )
    summary['anomaly_rate'] = summary['n_anomaly'] / summary['n_windows']
    display(summary.sort_values('anomaly_rate', ascending=False).head(20))

    first_location = scores['location'].iloc[0]
    selected = scores[scores['location'] == first_location]
    ax = selected.plot(x='window_end', y='anomaly_score', figsize=(14, 4), legend=False)
    ax.axhline(selected['threshold'].iloc[0], color='tab:red', linestyle='--', label='threshold')
    ax.set_title(f'MTGFlow – {first_location} – seed {SEED}')
    ax.set_ylabel('Anomaly score')
    ax.legend()
    plt.show()
else:
    print('CSV non trovato:', scores_path)

if metadata_path.is_file():
    run_metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    display(run_metadata)